# RAG (Retrieval Augmented Generation)

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

## 4) Store (ChromaDB)

In [4]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

## Store: Chroma DB를 사용
# uv add langchain-chroma chromadb
from langchain_chroma import Chroma

In [ ]:
# load
loader = DoclingLoader(
    file_path=['./data/wt_data.pdf'],
    export_type=ExportType.MARKDOWN
)
documents = loader.load()


# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)
doc_chunks = splitter.split_documents(documents)


# embed
embeddings = OpenAIEmbeddings()


# store
vector_store = Chroma.from_documents(   # embedding API가 여기서 호출됨
    documents=doc_chunks,               # 청크를 직접 넣을 수 있음
    embedding=embeddings,
    collection_name='school',
    persist_directory='./my_chroma_db'
)

[INFO] 2026-04-24 09:56:23,759 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-24 09:56:23,766 [RapidOCR] download_file.py:60: File exists and is valid: C:\potenup3\04-agent-langchain\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-24 09:56:23,767 [RapidOCR] main.py:57: Using C:\potenup3\04-agent-langchain\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-24 09:56:23,871 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-24 09:56:23,875 [RapidOCR] download_file.py:60: File exists and is valid: C:\potenup3\04-agent-langchain\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-24 09:56:23,876 [RapidOCR] main.py:57: Using C:\potenup3\04-agent-langchain\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-24 09:56:23,921 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-24 09:56:23,933 [R

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [9]:
# 저장된 ChromaDB 불러오기
vector_store = Chroma(
    collection_name='school',
    embedding_function=embeddings,
    persist_directory='./my_chroma_db'
)

In [10]:
# 중간 확인 과정: 만일 유사한 정보가 나오지 않았다면 split단계나 임베딩을 바꿔봐야 한다.
results = vector_store.similarity_search('상담예약 및 취소 규정에 대해 알려줘.', k=3) # k는 결과 개수
results

[Document(id='cb2add50-aa99-4470-adf6-54926dea48f3', metadata={'source': './data/wt_data.pdf'}, page_content='| 항목      | 기준                             | 조치                               |\n|-----------|----------------------------------|------------------------------------|\n| 일반취소  | 상담시 작 4시간전까지            | 불 이 익없음                       |\n| 지연취소  | 상담시 작 4시간전이 후 ~시 작 전 | 월 2 회 까지 경 고 없음            |\n| 노쇼      | 사 전연락 없 이 미참석           | 해 당 학기 온 라인예약2 주 제한    |'),
 Document(id='ecab9474-1197-4fd4-b17a-5e51008d61ca', metadata={'source': './data/wt_data.pdf'}, page_content='| 노쇼      | 사 전연락 없 이 미참석           | 해 당 학기 온 라인예약2 주 제한    |\n| 반 복노쇼 | 같 은학기2 회 이상               | 상담 사사 전 승 인 후 에만예약가능 |'),
 Document(id='4b960f0a-4318-431f-8b29-07ef3a38329a', metadata={'source': './data/wt_data.pdf'}, page_content='## 2-1. 일반 상담 예약\n\n- 학업상담과 진로상담은 1 회 30분 기 준 이다.\n- 심리상담 초기면담은 50분 기 준 이며, 후속 회 기는 담 당 자 판단 에 따라 조정된다.\n- 하 루 에 동 일 학생이 예약할 수 있는 상담은 최 대 2 건 이다.\n- 예약 확정 문자는 상담 시 작 1시간 전에 자 동

In [ ]:
# ChromaDB에 추가로 저장할 때:
from langchain_core.documents import Document

new_docs = [
    Document(
        page_content="우리 학교의 대표 식당은 '모수'입니다.", 
        metadata={}         # 삭제할 때, 조회할 때 꼭 필요함
    )
]

vector_store.add_documents(new_docs)

['95a87959-5ef3-4958-ae29-2b42fdaa8cae']

In [15]:
vector_store.delete(ids=['95a87959-5ef3-4958-ae29-2b42fdaa8cae'])

In [16]:
collections = vector_store._collection
collections.count()

20

In [17]:
data = collections.get(
    limit=5, 
    include=["metadatas", "documents"]
)
data

{'ids': ['370662f4-bff1-464e-9b27-f99985ba721c',
  '037b9f91-ddbb-4393-98ef-30c4eb0d8900',
  'a0bac485-f37e-4253-96b3-a15542a8f884',
  '49bc060d-7105-4698-ab39-fc0551d9130c',
  '23c5b02e-1d1a-4211-a9b8-98bb70443115'],
 'embeddings': None,
 'documents': ['## RAG 실습용 PDF 예시\n\n학생지원센터 안내서 / Sample document for retrieval-augmented generation practice\n\n## 문서 목적\n\n이 문서는 RAG 실습을 위해 의도적으로 구성한 예시 자료입니다. FAQ, 정책, 절차, 연락처, 예외 규정, 비교 표를 함께 포함해 문서 로딩, 청킹, 임베딩, 검색, 답변 생성 실습에 활용할 수 있습니다.\n\n## 실습 포인트\n\n- 1) 제목과 소제목이 분명한 구조형 문서 2) 비슷하지만 다른 규정 문장 3) 표와 리스트가 섞인 반정형 데이터 4) 예외 조항이 포함된 정책형 문장\n\n## 권장 활용 예시',
  "- 1) 제목과 소제목이 분명한 구조형 문서 2) 비슷하지만 다른 규정 문장 3) 표와 리스트가 섞인 반정형 데이터 4) 예외 조항이 포함된 정책형 문장\n\n## 권장 활용 예시\n\n- 학생이 '상담 예약은 언제까지 취소할 수 있나요?'라고 물으면 정책 조항을 검색해 답하기\n- 장학금, 상담, 비교과 프로그램 등 여러 섹션 중 관련 영역만 정확히 검색하기\n- 표에 있는 운영시간과 본문에 있는 예외규정을 함께 결합해 답변하기\n- 청크 크기와 overlap에 따라 검색 품질이 어떻게 달라지는지 비교하기\n\n버전 1.0 / 발행일 2026-04-14 / 발행처 예시대학교 학생지원센터\n\n## 1. 센터 개요",
  '버전 1.0 / 발행일 2026-04-14 / 발행처 예시대학교 학생지원센터\n

## 5) Retrieve

In [ ]:
retriever = vector_store.as_retriever( # 기본적으로 similarity_search()와 기능은 같으나 langchain에서 제공함
    search_type='similarity' # 'mmr': Maximal Marginal Relevance == 다양성도 고려. 하지만 안쓰는게 좋음.
    search_kwargs={'k': 3}
)

retriever.invoke('상담시간 알려주세요.')

[Document(id='4b960f0a-4318-431f-8b29-07ef3a38329a', metadata={'source': './data/wt_data.pdf'}, page_content='## 2-1. 일반 상담 예약\n\n- 학업상담과 진로상담은 1 회 30분 기 준 이다.\n- 심리상담 초기면담은 50분 기 준 이며, 후속 회 기는 담 당 자 판단 에 따라 조정된다.\n- 하 루 에 동 일 학생이 예약할 수 있는 상담은 최 대 2 건 이다.\n- 예약 확정 문자는 상담 시 작 1시간 전에 자 동 발 송 된다.\n\n## 2-2. 취소 및 노쇼 정책'),
 Document(id='23c5b02e-1d1a-4211-a9b8-98bb70443115', metadata={'source': './data/wt_data.pdf'}, page_content='## 1-2. 위치와 연락처\n\n학생 회 관 2 층 204 호 에 위 치 하며, 대표전 화 는 02-000-1234, 대표 이 메 일은 support@example.ac.kr 이다. 심리상담실은 같 은 층 209 호 에 별 도로 있다.\n\n## 2. 상담 예약 및 취소 규정\n\n상담은 온 라인 예약 시스 템 을 통해 신청하는 것 을 원 칙 으로 하며, 당 일 방 문 접 수는 남 는 시간대가 있을 경우 에만 가능하다. 예약은 상담 시 작 2시간 전까지 할 수 있다.\n\n## 2-1. 일반 상담 예약'),
 Document(id='cb2add50-aa99-4470-adf6-54926dea48f3', metadata={'source': './data/wt_data.pdf'}, page_content='| 항목      | 기준                             | 조치                               |\n|-----------|----------------------------------|------------------------------------|\n| 일반